# Orca Core (Novus) — QLoRA Fine-Tune v1 (Kaggle, hardened)

Same hardened pipeline proven on Nano (T4-forced, disk-space-safe GGUF export via /tmp,
no mid-training checkpointing due to a pickling bug, adapter saved immediately after training)
— adapted for Core's base model and hyperparameters per `orca/train/variants.py`.

**Honest scope**: only 914 train / 49 eval examples right now (core's distillation run
produced 999 raw examples total, smaller than Nano's ~2200) — expect a lower quality
ceiling than Nano until more core distillation data exists. This run establishes a
real baseline to compare against, not a finished model.

Base model: `unsloth/Meta-Llama-3.1-8B-Instruct`, LoRA rank 64 (vs Nano's 16), batch
size 2 + grad accumulation 8, max_seq_length 8192 (vs Nano's 2048) — per Core's spec.

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "0"

# Plain PyPI install — no git+https source, no build-from-source step.
!pip install -q unsloth trl transformers datasets peft bitsandbytes accelerate

## Find your uploaded training data

Searches recursively under `/kaggle/input/` so it doesn't matter what your dataset was named.

In [ ]:
import glob

train_matches = glob.glob('/kaggle/input/**/orca_core_llama3_train_v2.jsonl', recursive=True)
eval_matches  = glob.glob('/kaggle/input/**/orca_core_llama3_eval_v2.jsonl', recursive=True)

print('Train file found:', train_matches)
print('Eval file found:', eval_matches)

if not train_matches:
    raise FileNotFoundError(
        "orca_core_llama3_train_v2.jsonl not found under /kaggle/input/. "
        "Make sure the dataset is attached via 'Add Input' in the right sidebar."
    )

train_path = train_matches[0]
eval_path = eval_matches[0] if eval_matches else None

In [ ]:
import json

def load_jsonl(path):
    lines = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    lines.append(json.loads(line))
                except Exception:
                    pass
    return lines

raw_train = load_jsonl(train_path)
raw_eval  = load_jsonl(eval_path) if eval_path else raw_train[:max(1, len(raw_train)//10)]
print(f'train={len(raw_train)} eval={len(raw_eval)}')

## Load base model (4-bit) + attach LoRA

Rank 64 — Core's spec (vs Nano's 16), sized for the balanced 8B model on a free-tier GPU.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192
base_model = "unsloth/Meta-Llama-3.1-8B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from datasets import Dataset

# The formatter.py output already has a 'text' field per example (llama3 format) — use it directly.
train_ds = Dataset.from_list([{"text": ex["text"]} for ex in raw_train])
eval_ds  = Dataset.from_list([{"text": ex["text"]} for ex in raw_eval])
print(f'train_ds={len(train_ds)} eval_ds={len(eval_ds)}')

## Train

Batch size 2 + grad accumulation 8 (effective batch 16, per Core's spec), 2 epochs.
`average_tokens_across_devices=False` prevents the known Unsloth/Transformers crash.
No mid-training checkpointing (`save_strategy="no"`) — hit a real Kaggle-side pickling
bug on the Nano runs; the adapter-save cell right after training is the real safety net.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import time

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_steps=50,
        save_strategy="no",
        output_dir="/kaggle/working/output",
        eval_strategy="steps",
        report_to="none",
        average_tokens_across_devices=False,
    ),
)

print("[train] starting QLoRA training...")
t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60
print(f"[train] done in {elapsed:.1f} min")

## Save the LoRA adapter immediately (before merge/export)

Protects the trained weights even if the merge/GGUF-export step crashes.

In [ ]:
adapter_dir = "/kaggle/working/adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"[adapter] saved to {adapter_dir} — trained weights are now safe on disk.")
!ls -la {adapter_dir}

## Merge LoRA + export GGUF (done in /tmp, not /kaggle/working)

`/kaggle/working/` has a hard 19.5GB quota — the merged 16-bit 8B model plus the
intermediate F16 GGUF exceed that. `/tmp` is a separate, larger scratch disk.

In [ ]:
import shutil

shutil.rmtree("/tmp/merged", ignore_errors=True)
shutil.rmtree("/tmp/gguf", ignore_errors=True)

print("[merge] merging LoRA adapters (in /tmp)...")
model.save_pretrained_merged("/tmp/merged", tokenizer, save_method="merged_16bit")
print("[merge] saved to /tmp/merged")

print("[gguf] converting to GGUF q4_k_m (in /tmp)...")
model.save_pretrained_gguf("/tmp/gguf", tokenizer, quantization_method="q4_k_m")
print("[gguf] saved under /tmp")

## Copy the final GGUF back to /kaggle/working/

Recursive + case-insensitive search since Unsloth's output folder naming varies.

In [ ]:
import glob, shutil, os

candidates = [f for f in glob.glob('/tmp/**/*.gguf', recursive=True) if 'q4_k_m' in f.lower()]
print('Found in /tmp:', candidates)

if candidates:
    source_path = candidates[0]
    filename = os.path.basename(source_path)
    dest_path = f'/kaggle/working/{filename}'
    shutil.copy(source_path, dest_path)
    print(f"[export] copied to {dest_path}")
    print("\nNext: click 'Save Version' -> 'Save & Run All (Commit)' at the top right.")
else:
    print('No GGUF file found under /tmp — check the [gguf] cell above for errors.')
    print('If training + adapter save both succeeded, your trained weights are still')
    print('safe in /kaggle/working/adapter — you can retry just this export cell.')